### Import

In [ ]:
from google.colab import drive
drive.mount('/content/gdrive')

Mounted at /content/gdrive


In [ ]:
data_path = "YOUR_PATH"

In [ ]:
import os

!pip uninstall -y langchain langchain-core langchain-community langchain-text-splitters langchain-upstage openai
!pip cache purge

req_path = os.path.join(data_path, 'requirements.txt')
!pip install -r "{req_path}"

Found existing installation: langchain 1.1.0
Uninstalling langchain-1.1.0:
  Successfully uninstalled langchain-1.1.0
Found existing installation: langchain-core 1.1.0
Uninstalling langchain-core-1.1.0:
  Successfully uninstalled langchain-core-1.1.0
Found existing installation: openai 2.8.1
Uninstalling openai-2.8.1:
  Successfully uninstalled openai-2.8.1
Files removed: 0
  Preparing metadata (setup.py) ... done
INFO: pip is looking at multiple versions of langchain-upstage to determine which version is compatible with other requirements. This could take a while.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 13.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 449.6/449.6 kB 30.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 51.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 60.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.7/23.7 MB 81.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [ ]:
import re
import time
import pickle
import json
import pandas as pd
import numpy as np
import requests
import html2text
import wikipediaapi
from bs4 import BeautifulSoup
from langchain.schema import Document
from langchain_upstage import UpstageEmbeddings, ChatUpstage
from langchain_text_splitters import Language, RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS
from langchain.retrievers import EnsembleRetriever
from langchain_community.retrievers import BM25Retriever
from langchain_core.prompts import PromptTemplate
from langchain.prompts import FewShotPromptTemplate, PromptTemplate as LP
from langchain_core.output_parsers import StrOutputParser
from openai import OpenAI
from datasets import load_dataset
from dotenv import load_dotenv
from tqdm import tqdm

### CONFIGURATION

In [ ]:
load_dotenv()

UPSTAGE_API_KEY = os.getenv('UPSTAGE_API_KEY')

if not UPSTAGE_API_KEY:
    raise ValueError("UPSTAGE_API_KEY가 설정되지 않았습니다. .env 파일을 확인하세요.")

embedding_path = os.path.join(data_path, "embeddings")
os.makedirs(embedding_path, exist_ok=True)

client = OpenAI(
    api_key=UPSTAGE_API_KEY,
    base_url="https://api.upstage.ai/v1"
)

### Load Embeddings

In [ ]:
upstage_embeddings = UpstageEmbeddings(
    api_key=UPSTAGE_API_KEY,
    model="solar-embedding-1-large"
)


# EWHA 임베딩 로드
print("【 EWHA 임베딩 로드 】\n")
ewha_embedding_path = os.path.join(embedding_path, "ewha-vectorstore")
ewha_bm25_path = os.path.join(embedding_path, "ewha_bm25_retriever.pkl")

ewha_vectorstore = FAISS.load_local(
    ewha_embedding_path,
    upstage_embeddings,
    allow_dangerous_deserialization=True
)
ewha_dense_retriever = ewha_vectorstore.as_retriever(search_kwargs={"k": 3})
print("EWHA Dense 임베딩 로드 완료")

with open(ewha_bm25_path, 'rb') as f:
    ewha_sparse_retriever = pickle.load(f)
print("EWHA BM25 로드 완료")

ewha_hybrid_retriever = EnsembleRetriever(
    retrievers=[ewha_dense_retriever, ewha_sparse_retriever],
    weights=[0.6, 0.4]
)
print("\nEWHA 하이브리드 검색 준비 완료 !")


# 학문 자료 임베딩 로드
print("\n【 학문 자료 임베딩 로드 】\n")
academic_embedding_path = os.path.join(embedding_path, "academic-vectorstore")
academic_bm25_path = os.path.join(embedding_path, "academic_bm25_retriever.pkl")

academic_vectorstore = FAISS.load_local(
    academic_embedding_path,
    upstage_embeddings,
    allow_dangerous_deserialization=True
)
academic_dense_retriever = academic_vectorstore.as_retriever(search_kwargs={"k": 3})
print("학문 자료 Dense 임베딩 로드 완료")

with open(academic_bm25_path, 'rb') as f:
    academic_sparse_retriever = pickle.load(f)
print("학문 자료 BM25 로드 완료")

academic_hybrid_retriever = EnsembleRetriever(
    retrievers=[academic_dense_retriever, academic_sparse_retriever],
    weights=[0.6, 0.4]
)
print("\n학문 자료 하이브리드 검색 준비 완료 !")

【 EWHA 임베딩 로드 】

EWHA Dense 임베딩 로드 완료
EWHA BM25 로드 완료

EWHA 하이브리드 검색 준비 완료 !

【 학문 자료 임베딩 로드 】

학문 자료 Dense 임베딩 로드 완료
학문 자료 BM25 로드 완료

학문 자료 하이브리드 검색 준비 완료 !


## 2-Stage LLM

In [ ]:
# @title 1-1. LLM 초기화

llm_classifier = ChatUpstage(api_key=UPSTAGE_API_KEY, model="solar-pro2") # Stage 1: 카테고리 분류용
llm_reasoning = ChatUpstage(api_key=UPSTAGE_API_KEY, model="solar-pro2")  # Stage 2: 답변 생성용

In [ ]:
# @title 1-2. Instruction & Few-Shot Database

# category별 instruction 정의
CATEGORY_INSTRUCTIONS = {
    "business": (
        "Apply standard micro/macroeconomics, finance, and management theories.\n"
        "Focus on profit maximization, market equilibrium, and strategic rationality."
    ),
    "law": (
        "Apply legal doctrines, statutes, and case-law principles provided in the context.\n"
        "Focus on elements of the crime/tort, standing, and strict statutory interpretation."
    ),
    "history": (
        "Analyze chronological events, causal relationships, and historical context.\n"
        "Avoid anachronisms; focus on the specific era and documented historical facts."
    ),
    "psychology": (
        "Apply clinical definitions, cognitive theories, and experimental findings.\n"
        "Focus on diagnostic criteria, behavioral mechanisms, and neurotransmitter functions."
    ),
    "philosophy": (
        "Analyze the logical structure, premises, and validity of the arguments.\n"
        "Focus on definitions of philosophical terms and strict logical deduction."
    ),
    "general": (
        "Use standard academic reasoning and logical deduction based on the context."
    )
}


FEW_SHOT_DATABASE = {
    "korean_regulation": [
        {
            "question": "휴학기간은 통산하여 a년을 초과할 수 없는데, 예외적으로 건축학전공은 b년을 초과할 수 없다. a와 b가 상수일 때 a+b의 값을 구하면?",
            "context": "제26조 ④ 휴학기간은 통산하여 3년(건축학전공의 경우 4년, 의예과의 경우 3학기)을 초과할 수 없다.",
            "reasoning": "제26조 4항에 휴학기간은 통산하여 3년을 초과할 수 없다고 명시되어 있으므로 a=3이고, 건축학전공은 4년을 초과할 수 없으므로 b=4이다. 따라서 a+b=7이다.",
            "answer": "(D)"
        }
    ],
    "business": [
        {
            "question": "If the price elasticity of demand for a product is greater than 1 (elastic), which outcome is most likely when the firm lowers price?",
            "context": (
                "Price elasticity of demand measures the percentage change in quantity demanded divided by the percentage change in price. "
                "If elasticity > 1, quantity demanded responds proportionally more than price. Total revenue equals price times quantity. "
                "When demand is elastic, a decrease in price leads to a proportionally larger increase in quantity demanded, raising total revenue."
            ),
            "reasoning": (
                "Apply the definition: elasticity > 1 implies quantity change > price change in percent terms. "
                "Since TR = P * Q, a price cut increases Q enough that TR increases. "
                "Therefore the option stating total revenue will increase is correct."
            ),
            "answer": "[ANSWER]: (A)"
        }
    ],
    "law": [
        {
            "question": "In contract law, which option best describes valid consideration?",
            "context": (
                "Consideration is a bargained-for exchange between parties. "
                "Each party must either confer a legal benefit or suffer a legal detriment. "
                "A promise to do what one is already legally obligated to do is not valid consideration, "
                "because it adds no new legal detriment or benefit."
            ),
            "reasoning": (
                "Use the definition of consideration: there must be a new legal detriment or benefit. "
                "Options that merely restate an existing duty are not valid consideration. "
                "Therefore, the correct choice is the one that describes a bargained-for exchange "
                "where each party incurs a new legal detriment or benefit."
            ),
            "answer": "[ANSWER]: (C)"
        }
    ],
    "history": [
        {
            "question": "Which explanation best identifies the immediate catalyst that led to the outbreak of World War I in 1914?",
            "context": (
                "Historical analyses distinguish long-term structural causes (militarism, alliances, imperial rivalries) "
                "from immediate catalysts. The assassination of Archduke Franz Ferdinand by a Bosnian nationalist in Sarajevo "
                "served as the triggering event that activated alliance commitments and mobilization plans, rapidly escalating a regional crisis into a continent-wide war."
            ),
            "reasoning": (
                "The context separates structural causes from the immediate trigger. While alliances and militarism set the stage, "
                "the assassination directly prompted Austria-Hungary's ultimatum to Serbia and the consequent chain of mobilizations. "
              "Thus the correct option identifies the assassination and the resulting diplomatic-military cascade as the immediate catalyst."
            ),
            "answer": "[ANSWER]: (B)"
        },
    ],
    "psychology": [
        {
            "question": "In classical conditioning, Pavlov’s dogs learned to salivate to the sound of a bell after the bell had been repeatedly paired with food. What best labels the salivation that occurs in response to the bell?",
            "context": (
                "In Pavlovian (classical) conditioning, an unconditioned stimulus (US) elicits an unconditioned response (UR) naturally. "
                "A neutral stimulus (NS) paired repeatedly with the US can become a conditioned stimulus (CS) that elicits a conditioned response (CR). "
                "After conditioning, the response to the CS is called the conditioned response."
            ),
            "reasoning": (
                "Given the context, salivation originally was an unconditioned response to food (US → UR). "
                "After pairing, the bell becomes a conditioned stimulus and the salivation to the bell is the conditioned response. "
                "Thus the correct option names this as the conditioned response."
            ),
            "answer": "[ANSWER]: (D)"
        },
    ],
    "philosophy": [
        {
            "question": "Which statement best captures the core claim of classical act-utilitarian moral theory?",
            "context": (
                "Act utilitarianism holds that the rightness of an action depends solely on the consequences of that specific action, "
                "measured by the net balance of pleasure over pain (or overall welfare). Moral agents should choose the action that maximizes aggregate utility, "
                "impartial to personal attachments."
            ),
            "reasoning": (
                "The context defines act utilitarianism as consequence-based and welfare-maximizing for each act. "
                "Therefore the correct option is the one asserting that the morally right action is the one that produces the greatest total welfare for those affected."
            ),
            "answer": "[ANSWER]: (C)"
        }
    ],
    "general": [
        {
            "question": "In contract law, which option best describes valid consideration?",
            "context": (
                "Consideration is a bargained-for exchange between parties. "
                "Each party must either confer a legal benefit or suffer a legal detriment. "
                "A promise to do what one is already legally obligated to do is not valid consideration, "
                "because it adds no new legal detriment or benefit."
            ),
            "reasoning": (
                "Use the definition of consideration: there must be a new legal detriment or benefit. "
                "Options that merely restate an existing duty are not valid consideration. "
                "Therefore, the correct choice is the one that describes a bargained-for exchange "
                "where each party incurs a new legal detriment or benefit."
            ),
            "answer": "[ANSWER]: (C)"
        }
    ]
}

PROMPT_TEMPLATES = {
    "en": {
        "header": "[Example - {category}]",
        "question": "Question",
        "context": "Context",
        "reasoning": "Reasoning",
        "answer": "Answer"
    },
    "ko": {
        "header": "[예시]",
        "question": "질문",
        "context": "문맥",
        "reasoning": "추론",
        "answer": "정답"
    }
}

def get_few_shot_text(category: str, lang: str = "en"):
    example = FEW_SHOT_DATABASE.get(category,[])
    labels = PROMPT_TEMPLATES.get(lang, PROMPT_TEMPLATES["en"])
    formatted_text = []

    if not example:
        return ""

    for ex in example:
        block = (
            f"\n{labels['header'].format(category=category.upper())}\n"
            f"{labels['question']}: {ex.get('question', '')}\n"
            f"{labels['context']}: {ex.get('context', '')}\n"
            f"{labels['reasoning']}: {ex.get('reasoning', '')}\n"
            f"{labels['answer']}: {ex.get('answer', '')}\n"
        )
        formatted_text.append(block)

    return "\n".join(formatted_text)

In [ ]:
# @title 1-3. [Stage 1] Multi-label 분류기 (Category Classifier)

VALID_CATEGORIES = ['business', 'history', 'law', 'philosophy', 'psychology']


def parse_multilabel_output(text):
    """
    LLM 출력에서 Primary와 Secondary 카테고리를 파싱
    형식 예: "PRIMARY: law, SECONDARY: history"
    """
    text = text.lower()

    primary = "general"
    secondary = None

    # 정규식으로 추출
    p_match = re.search(r"primary\s*:\s*([a-z_]+)", text, re.IGNORECASE)
    s_match = re.search(r"secondary\s*:\s*([a-z_]+|none)", text, re.IGNORECASE)

    if p_match:
        val = p_match.group(1).lower()
        if val in VALID_CATEGORIES:
            primary = val

    if s_match:
        val = s_match.group(1).lower()
        if val in VALID_CATEGORIES:
            secondary = val
        elif val == 'none':
            secondary = None

    # 예외 처리
    if "ewha" in text or re.search(r'[가-힣]', text):
        return "korean_regulation", None

    return primary, secondary


def get_category_classifier_chain_multilabel():

    template = """Analyze the following question and identify the top 2 most relevant academic categories.

Valid Categories: {categories}

[Instructions]
1. **Primary Category**: The main discipline the question belongs to.
2. **Secondary Category**: A related discipline that might provide useful context or reasoning rules (e.g., A Law question might have Ethical/Philosophy aspects).
3. If the question purely belongs to one domain, set Secondary to 'None'.
4. Output Format must be EXACTLY:
   PRIMARY: [category_name]
   SECONDARY: [category_name]

Question: {question}

Analysis:"""

    prompt = PromptTemplate(
        template=template,
        input_variables=["question"],
        partial_variables={"categories": ", ".join(VALID_CATEGORIES)}
    )

    return prompt | llm_classifier | StrOutputParser()


In [ ]:
# @title 1-4. [Stage 2] Mixture Prompt Generator

def get_focused_rag_prompt(primary_cat: str, secondary_cat: str, num_choices: int):
    """
    Secondary Category를 입력받지만, 프롬프트 생성 시에는 무시
    """

    answer_options = "(A, B, C, D)" if num_choices == 4 else "(A, B, C, D, E, F, G, H, I, J)"

    # [CASE A] 한국어 규정 문제
    if primary_cat == "korean_regulation":
        few_shot_text = get_few_shot_text("korean_regulation", lang="ko")

        return PromptTemplate.from_template(f"""당신은 객관식 문제 답변 전문가입니다.

이 문제는 **이화여자대학교 학칙(EWHA.pdf)**을 바탕으로 출제된 문제입니다.

[절대 준수할 규칙]

1. **반드시 EWHA.pdf에 있는 문장과 숫자만** 사용해서 판단합니다.
   - 다른 대학의 학칙, 일반 상식, 법학/교육학 지식 등은 절대 사용하지 마세요.
   - 애매할 때도 추측하지 말고, **EWHA.pdf 문장과 표의 숫자**를 최우선으로 따르세요.
   - 당신의 배경지식보다, EWHA.pdf 문맥과 시험에서 일반적으로 사용하는 정의를 우선하세요.

2. 선택지를 고를 때는 다음 순서를 따릅니다.
   1) 먼저 EWHA.pdf에서 **문제와 직접 연결되는 조항(조문 번호, 문장, 표의 숫자 등)**을 정리합니다.
   2) 각 선택지가 **그 조문을 얼마나 정확히 반영하는지** 비교합니다.
      - 문구, 숫자, 조건(예외 조항 포함)이 조문과 가장 비슷한 것을 우선합니다.
   3) 여러 선택지가 가능해 보이면, **조문과 정반대이거나, 조문에 없는 내용을 가정하는 선택지부터 오답으로** 처리합니다.

3. 문제에 “해당하지 않는 것”, “제외되는 것”, “제적되지 않는 경우”처럼
   **부정형 표현**이 있을 때는 반드시 아래 순서를 따릅니다.
   - (1) 먼저 EWHA.pdf 문맥에서 **해당/제적/인정 사유** 전체를 정리합니다.
   - (2) 그 목록에 **포함되는 선택지**를 먼저 표시합니다.
   - (3) 그 목록에 **포함되지 않거나, 예외 규정에 해당하는 선택지**를 답으로 선택합니다.

4. 숫자·연한·학점 관련 문제의 경우:
   - 문제에서 묻는 조건(예: “졸업 최소 기준”, “조기졸업 기준”, “통산 휴학 가능 연한”)에
     **가장 정확히 대응하는 숫자**를 고릅니다.
   - 상식적 추론을 하지 말고, **조문에 적힌 그대로** 사용합니다.

문제: {{question}}
EWHA.pdf 문맥: {{context}}

{few_shot_text}

다음 형식과 순서를 꼭 지켜서 생각하고 답변하세요.

단계 1: [EWHA.pdf에서 발견한 관련 정보]
- 문제와 직접적으로 연결되는 조항(조문 번호, 문장, 표의 숫자 등)을 요약합니다.
- 부정형 문제라면, 먼저 학칙에서 규정한 "해당/제적/인정" 조건 전체를 정리합니다.

단계 2: [추론 과정]
- 위 조항을 바탕으로, 문제에서 묻는 조건을 정확히 해석합니다.
- 특히, 최소 기준인지(“…이상”), 최대 한도인지(“…초과할 수 없다”), 예외 조항인지(특정 전공/휴학 등)를 명확히 정리합니다.
- 일반 상식이나 다른 학교 사례를 사용하지 말고, **EWHA.pdf 문장만으로** 논리를 전개합니다.

단계 3: [선택지 비교]
- 각 선택지 {answer_options}가 EWHA.pdf 조문과 어떻게 다른지/같은지 하나씩 비교합니다.
- 조문과 **가장 일치하는 선택지** 또는
  부정형 문제의 경우 **조문에 규정되지 않은/예외에 해당하는 선택지**를 찾습니다.
- 시험 문제답게, 여러 해석이 가능해 보이면
  ① 조문과 정반대인 선택지,
  ② 조문에 전혀 언급되지 않은 내용을 추가로 가정하는 선택지부터 오답으로 처리합니다.

최종 답변: [ANSWER]: (X)  (X는 {answer_options} 중 하나)""")


    # [CASE B] 영어 MMLU 문제
    else:
        few_shot_text = get_few_shot_text(primary_cat, lang="en")

        return PromptTemplate.from_template(f"""You are an expert at answering difficult multiple-choice questions (MMLU-Pro style), specializing in **{primary_cat.upper()}**.

Your job:
- Use the **academic context** below as your PRIMARY source.
- BUT you may also use your own background knowledge **to fill in gaps or break ties**.
- If the context gives a **clear definition, rule, or condition** related to **{primary_cat.upper()}**,
  you MUST follow that strictly, even if it conflicts with your memory.

Question: {{question}}

Academic Context: {{context}}

{few_shot_text}

Guidelines:
1. First, extract the key definitions, rules, or facts from the context that specifically relate to **{primary_cat.upper()}**.
2. Then check how each option {answer_options} matches or conflicts with those facts.
3. If the context is vague or incomplete, rely ONLY on standard **textbook knowledge of {primary_cat.upper()}** to choose. Do not mix concepts from other disciplines.
4. Prefer the option that:
   - Best matches the core **{primary_cat.upper()}** principle in the context, AND
   - Sounds like a typical textbook or exam answer.
5. Avoid speculative or overly creative interpretations.

Format your response exactly as follows:

Step 1: [Key concepts and relevant information]
Step 2: [Understanding the question's requirements]
Step 3: [Option analysis]
Final Answer: [ANSWER]: (X) where X is one of {answer_options}""")


In [ ]:
# @title 1-5. 정답 추출기

def extract_answer(response_text: str):
    if re.search(r'[가-힣]', str(response_text)):
        valid_chars = "ABCD"
    else:
        valid_chars = "ABCDEFGHIJ"

    patterns = [
        r"최종 답변:[\s\*]*\[ANSWER\]:[\s\*]*[\(\[\{]?([A-J])",
        r"\*\*ANSWER\*\*:[\s\*]*[\(\[\{]?([A-J])",
        r"\[ANSWER\]:[\s\*]*[\(\[\{]?([A-J])",
        r"Final Answer:[\s\*]*[\(\[\{]?([A-J])",
        r"Answer:[\s\*]*[\(\[\{]?([A-J])",
        r"ANSWER:[\s\*]*[\(\[\{]?([A-J])",
        r"정답:[\s\*]*[\(\[\{]?([A-J])",

        # LaTeX 포맷
        r"\\boxed\s*\{?\s*\(?([A-J])",

        # 문장 맨 끝에 답이 나오는 경우
        r"[\s\*]*[\(\[\{]([A-J])[\)\]\}][\s\*]*$"
    ]

    candidates = []

    for pattern in patterns:
        for match in re.finditer(pattern, response_text, re.IGNORECASE):
            ans = match.group(1).upper()
            if ans in valid_chars:
                # (등장 위치 index, 정답 문자) 저장
                candidates.append((match.start(), ans))

    if candidates:
        candidates.sort(key=lambda x: x[0]) # 위치순 정렬
        return candidates[-1][1] # 가장 뒤에 있는 정답 채택

    return None

In [ ]:
# @title 1-6. 쿼리 강화

def create_enhanced_query(prompts_text):
    # 줄바꿈 문자를 공백으로 치환하여 한 줄로 만듦
    clean_text = prompts_text.replace('\n', ' ').replace('\r', ' ')

    # 불필요한 반복 공백 제거
    enhanced_query = re.sub(r'\s+', ' ', clean_text).strip()

    return enhanced_query

In [ ]:
# @title 2. 데이터 로드

def load_data(file_path):
    """testset.csv 로드"""
    data = pd.read_csv(file_path)
    prompts = data['prompts']
    answers = data['answers']
    return prompts, answers

prompts, answers = load_data(os.path.join(data_path, 'data/testset.csv'))
print(f"테스트셋 로드: {len(prompts)}개 문제\n")

df = pd.DataFrame({'prompts': prompts, 'answers': answers})

테스트셋 로드: 50개 문제



In [ ]:
# @title 3. [Stage 1] Batch Category Classification (Multi-label)

print("=" * 70)
print("[Stage 1] Multi-label Classification Started...")
print("=" * 70)

classifier_chain = get_category_classifier_chain_multilabel()
pred_primary = []
pred_secondary = []
detected_languages = []

for question in tqdm(prompts, desc="Classifying"):
    if re.search(r'[가-힣]', str(question)):
        lang = "ko"
        p_cat, s_cat = "korean_regulation", None
    else:
        lang = "en"
        try:
            raw_out = classifier_chain.invoke({"question": question}) # LLM 호출
            p_cat, s_cat = parse_multilabel_output(raw_out) # 파싱
        except Exception as e:
            print(f"⚠️ Classification Error: {e}")
            p_cat, s_cat = "general", None

    detected_languages.append(lang)
    pred_primary.append(p_cat)
    pred_secondary.append(s_cat)


df['pred_primary'] = pred_primary
df['pred_secondary'] = pred_secondary
df['language'] = detected_languages

print("\n\nClassification Complete !")

[Stage 1] Multi-label Classification Started...


Classifying: 100%|██████████| 50/50 [00:56<00:00,  1.14s/it]



Classification Complete !


In [ ]:
# @title 4. [Stage 2] Mixture RAG Execution

print("=" * 70)
print("[Stage 2] Mixture RAG Generation Started...")
print(" - Using predefined Primary & Secondary categories from Stage 1")
print("=" * 70)

results = []
results_for_analysis = []

for idx, row in tqdm(df.iterrows(), total=len(df), desc="Processing"):
    question = row['prompts']

    # 1. Multi-label 카테고리 가져오기
    p_cat = row['pred_primary']
    s_cat = row['pred_secondary']
    language = row['language']
    correct_answer = row['answers']

    # 2. Retriever 설정
    if language == "ko":
        retriever = ewha_hybrid_retriever
        num_choices = 4
        search_query = create_enhanced_query(question)
    else:
        retriever = academic_hybrid_retriever
        num_choices = 10
        search_query = question

    # 3. Context 검색
    try:
        retrieved_docs = retriever.invoke(search_query)
        context_text = "\n\n".join([doc.page_content for doc in retrieved_docs])
    except Exception as e:
        print(f"Retrieval Error at idx {idx}: {e}")
        context_text = ""

    # 4. Mixture Prompt 생성
    prompt_template = get_focused_rag_prompt(p_cat, s_cat, num_choices)
    rag_chain = prompt_template | llm_reasoning | StrOutputParser()

    # 5. 답변 생성
    try:
        response_text = rag_chain.invoke({
            "question": question,
            "context": context_text
        })
    except Exception as e:
        print(f"Error in answer-generation: {e}")
        response_text = "Error"

    # 6. 정답 추출
    generated_answer = extract_answer(response_text)

    # 6. 결과 수집
    results.append({
        "question": str(question),
        "your_answer": generated_answer
    })

    results_for_analysis.append({
        "question_idx": idx,
        "question_snippet": str(question)[:100],
        "pred_primary": p_cat,
        "pred_secondary": s_cat,
        "correct_answer": correct_answer,
        "generated_answer": generated_answer,
        "full_response": response_text,
        "language": language
    })


print("\n\nRAG Generation Complete !")

[Stage 2] Mixture RAG Generation Started...
 - Using predefined Primary & Secondary categories from Stage 1


Processing: 100%|██████████| 50/50 [06:18<00:00,  7.57s/it]



RAG Generation Complete !


In [ ]:
# @title 5. 결과 저장

# 결과를 DataFrame으로 변환
results_df = pd.DataFrame(results)

# 파일 저장
save_path = os.path.join(data_path, "results/7_final.csv")
results_df.to_csv(save_path, index=False, encoding='utf-8-sig')
print(f"\nResults saved to: {save_path}")


Results saved to: /content/gdrive/MyDrive/Colab Notebooks/NLP-project/NLP-for-git/7_final.csv


In [ ]:
results_for_analysis_df = pd.DataFrame(results_for_analysis)

answers = results_for_analysis_df['correct_answer']
responses = results_for_analysis_df['full_response']

In [ ]:
# @title Print Accuracy

cnt = 0

for answer, response in zip(answers, responses):
    print("-"*10)
    generated_answer = extract_answer(response)
    print(response)
    # check
    if generated_answer:
        print(f"generated answer: {generated_answer}, answer: {answer}")
    else:
        print("extraction fail")

    if generated_answer == None:
        continue
    if generated_answer in answer:
        cnt += 1

print()
print(f"acc: {(cnt/len(answers))*100}%")

----------
단계 1: [EWHA.pdf에서 발견한 관련 정보]  
- 문제: 재학 중인 학생이 휴학을 신청해야 하는 기한은 학기 개시일로부터 며칠 이내인가?  
- 관련 조항: **제26조(휴학) ⑥ 재학 중인 자가 휴학을 하고자 하는 경우 학기개시일로부터 90일 이내에 휴학을 신청하여야 한다.**  

단계 2: [추론 과정]  
- 제26조 ⑥에 따르면, **학기 개시일로부터 90일 이내**에 휴학 신청을 해야 한다고 명시되어 있습니다.  
- 다른 조항(예: 제26조 ④, ⑦ 등)은 휴학 기간, 예외 사유, 신입생 제한 등을 다루지만, **휴학 신청 기한과는 무관**합니다.  
- 따라서 **90일**이 정답이며, 이는 선택지 (D)에 해당합니다.  

단계 3: [선택지 비교]  
- (A) 30일: **조문에 없는 수치**로 오답.  
- (B) 45일: **조문에 없는 수치**로 오답.  
- (C) 60일: **조문에 없는 수치**로 오답.  
- (D) 90일: **제26조 ⑥과 정확히 일치**하므로 정답.  

최종 답변: [ANSWER]: (D)
generated answer: D, answer: (D)
----------
단계 1: [EWHA.pdf에서 발견한 관련 정보]  
- 제31조(재입학) ② 재입학은 **1회**에 한하여 할 수 있다. 다만, 제28조제4호에 의하여 제적된 자는 제적된 날부터 **1년**이 경과한 후 재입학할 수 있다.  
- 문제에서 언급된 **a**는 "재입학 가능 횟수"를 의미하며, **b**는 "제28조제4호 제적자의 재입학 가능 기간"을 의미합니다.  
  - a = 1 (재입학 1회 한정)  
  - b = 1 (제적된 날부터 1년 경과 후 재입학)  

단계 2: [추론 과정]  
- **a**와 **b**는 각각 **1**과 **1**로, 상수입니다.  
- 따라서 **a + b = 1 + 1 = 2**입니다.  
- 문제의 조건과 선택지 중 **(A) 2**가 정확히 일치합니다.  

단계 